In [1]:
import os
import torch
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from torch import nn, optim
from collections import Counter
from tqdm.auto import tqdm
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter
from torchvision import transforms
from torchvision.datasets import VOCDetection

utils

In [ ]:
#计算iou return[n,1]
def iou_cal(boxes_preds, boxes_labels, box_format="midpoint"):
    """
    Calculates intersection over union

    Parameters:
        boxes_preds (tensor): Predictions of Bounding Boxes (BATCH_SIZE, 4)
        boxes_labels (tensor): Correct labels of Bounding Boxes (BATCH_SIZE, 4)
        box_format (str): midpoint/corners, if boxes (x,y,w,h) or (x1,y1,x2,y2)

    Returns:
        tensor: Intersection over union for all examples
    """

    if box_format == "midpoint":
        #0：1可能保留最后一个维度，[0]不能保存最后一个维度
        box1_x1 = boxes_preds[..., 0:1] - boxes_preds[..., 2:3] / 2
        box1_y1 = boxes_preds[..., 1:2] - boxes_preds[..., 3:4] / 2
        box1_x2 = boxes_preds[..., 0:1] + boxes_preds[..., 2:3] / 2
        box1_y2 = boxes_preds[..., 1:2] + boxes_preds[..., 3:4] / 2
        box2_x1 = boxes_labels[..., 0:1] - boxes_labels[..., 2:3] / 2
        box2_y1 = boxes_labels[..., 1:2] - boxes_labels[..., 3:4] / 2
        box2_x2 = boxes_labels[..., 0:1] + boxes_labels[..., 2:3] / 2
        box2_y2 = boxes_labels[..., 1:2] + boxes_labels[..., 3:4] / 2

    if box_format == "corners":
        box1_x1 = boxes_preds[..., 0:1]
        box1_y1 = boxes_preds[..., 1:2]
        box1_x2 = boxes_preds[..., 2:3]
        box1_y2 = boxes_preds[..., 3:4]  # (N, 1)
        box2_x1 = boxes_labels[..., 0:1]
        box2_y1 = boxes_labels[..., 1:2]
        box2_x2 = boxes_labels[..., 2:3]
        box2_y2 = boxes_labels[..., 3:4]

    x1 = torch.max(box1_x1, box2_x1)
    y1 = torch.max(box1_y1, box2_y1)
    x2 = torch.min(box1_x2, box2_x2)
    y2 = torch.min(box1_y2, box2_y2)

    # .clamp(0) is for the case when they do not intersect
    intersection = (x2 - x1).clamp(0) * (y2 - y1).clamp(0)

    box1_area = abs((box1_x2 - box1_x1) * (box1_y2 - box1_y1))
    box2_area = abs((box2_x2 - box2_x1) * (box2_y2 - box2_y1))

    #防止除0
    return intersection / (box1_area + box2_area - intersection + 1e-6)
#进行nms筛选输入list[list],返回list[list]
def nms(bboxes, iou_threshold, threshold, box_format="corners"):
    """
    Does Non Max Suppression given bboxes

    Parameters:
        bboxes (list): list of lists containing all bboxes with each bboxes
        specified as [class_pred, prob_score, x1, y1, x2, y2]
        iou_threshold (float): threshold where predicted bboxes is correct
        threshold (float): threshold to remove predicted bboxes (independent of IoU) 
        box_format (str): "midpoint" or "corners" used to specify bboxes

    Returns:
        list: bboxes after performing NMS given a specific IoU threshold
    """

    assert type(bboxes) == list

    bboxes = [box for box in bboxes if box[1] > threshold]
    bboxes = sorted(bboxes, key=lambda x: x[1], reverse=True)
    bboxes_after_nms = []

    while bboxes:
        chosen_box = bboxes.pop(0)

        bboxes = [
            box
            for box in bboxes
            if box[0] != chosen_box[0]
            or iou_cal(
                torch.tensor(chosen_box[2:]),
                torch.tensor(box[2:]),
                box_format=box_format,
            )
            < iou_threshold
        ]

        bboxes_after_nms.append(chosen_box)

    return bboxes_after_nms
#计算map（准确率与召回率种类积分的平均）
def mean_average_precision(pred_boxes, true_boxes,
                            iou_threshold=0.5, box_format="midpoint", num_classes=20
):
    """
    Calculates mean average precision 

    Parameters:
        pred_boxes (list): list of lists containing all bboxes with each bboxes
        specified as [image_idx, class_prediction, prob_score, x1, y1, x2, y2]
        true_boxes (list): Similar as pred_boxes except all the correct ones 
        iou_threshold (float): threshold where predicted bboxes is correct
        box_format (str): "midpoint" or "corners" used to specify bboxes
        num_classes (int): number of classes

    Returns:
        float: mAP value across all classes given a specific IoU threshold 
    """

    # list storing all AP for respective classes
    average_precisions = []

    # used for numerical stability later on
    epsilon = 1e-6

    for c in range(num_classes):
        detections = []
        ground_truths = []

        # Go through all predictions and targets,
        # and only add the ones that belong to the
        # current class c
        for detection in pred_boxes:
            if detection[1] == c:
                detections.append(detection)

        for true_box in true_boxes:
            if true_box[1] == c:
                ground_truths.append(true_box)

        # find the amount of bboxes for each training example
        # Counter here finds how many ground truth bboxes we get
        # for each training example, so let's say img 0 has 3,
        # img 1 has 5 then we will obtain a dictionary with:
        # amount_bboxes = {0:3, 1:5}
        amount_bboxes = Counter([gt[0] for gt in ground_truths])

        # We then go through each key, val in this dictionary
        # and convert to the following (w.r.t same example):
        # ammount_bboxes = {0:torch.tensor[0,0,0], 1:torch.tensor[0,0,0,0,0]}
        for key, val in amount_bboxes.items():
            amount_bboxes[key] = torch.zeros(val)

        # sort by box probabilities which is index 2
        detections.sort(key=lambda x: x[2], reverse=True)
        TP = torch.zeros((len(detections)))
        FP = torch.zeros((len(detections)))
        total_true_bboxes = len(ground_truths)
        
        # If none exists for this class then we can safely skip
        if total_true_bboxes == 0:
            continue

        for detection_idx, detection in enumerate(detections):
            # Only take out the ground_truths that have the same
            # training idx as detection
            ground_truth_img = [
                bbox for bbox in ground_truths if bbox[0] == detection[0]
            ]

            best_iou = 0

            for idx, gt in enumerate(ground_truth_img):
                iou = iou_cal(
                    torch.tensor(detection[3:]),
                    torch.tensor(gt[3:]),
                    box_format=box_format,
                )

                if iou > best_iou:
                    best_iou = iou
                    best_gt_idx = idx

            if best_iou > iou_threshold:
                # only detect ground truth detection once
                if amount_bboxes[detection[0]][best_gt_idx] == 0:
                    # true positive and add this bounding box to seen
                    TP[detection_idx] = 1
                    amount_bboxes[detection[0]][best_gt_idx] = 1
                else:
                    FP[detection_idx] = 1

            # if IOU is lower then the detection is a false positive
            else:
                FP[detection_idx] = 1

        TP_cumsum = torch.cumsum(TP, dim=0)
        FP_cumsum = torch.cumsum(FP, dim=0)
        recalls = TP_cumsum / (total_true_bboxes + epsilon)
        precisions = torch.divide(TP_cumsum, (TP_cumsum + FP_cumsum + epsilon))
        #在前面加上一个1
        precisions = torch.cat((torch.tensor([1]), precisions))
        recalls = torch.cat((torch.tensor([0]), recalls))
        # torch.trapz for numerical integration
        average_precisions.append(torch.trapz(precisions, recalls))

    return sum(average_precisions) / len(average_precisions)
#显式绘制框
def plot_image(image, boxes):
    """Plots predicted bounding boxes on the image"""
    im = np.array(image)
    height, width, _ = im.shape

    # Create figure and axes
    fig, ax = plt.subplots(1)
    # Display the image
    ax.imshow(im)

    # box[0] is x midpoint, box[2] is width
    # box[1] is y midpoint, box[3] is height

    # Create a Rectangle potch
    for box in boxes:
        class_pred = int(box[0])
        confidence = float(box[1])
        class_name = VOC_CLASSES[class_pred]
        label = f"{class_name}: {confidence:.2f}"

        box = box[2:]
        assert len(box) == 4, "Got more values than in x, y, w, h, in a box!"
        upper_left_x = box[0] - box[2] / 2
        upper_left_y = box[1] - box[3] / 2
        ax.text(upper_left_x * width,
            upper_left_y * height,
            label,
            fontsize=10,
            bbox=dict(facecolor="white",alpha=0.7,edgecolor="none",
            ),
        )

        rect = patches.Rectangle(
            (upper_left_x * width, upper_left_y * height),
            box[2] * width,
            box[3] * height,
            linewidth=1,
            edgecolor="r",
            facecolor="none",
        )
        # Add the patch to the Axes
        ax.add_patch(rect)

    plt.show()
#筛去数据集中，并加上image_idx
#  返回为[all_pred_boxes, all_true_boxes]
def get_bboxes(
    loader,
    model,
    iou_threshold,
    threshold,
    pred_format="cells",
    box_format="midpoint",
    device="cuda",
):
    all_pred_boxes = []
    all_true_boxes = []

    # make sure model is in eval before get bboxes
    model.eval()
    image_idx = 0

    for batch_idx, (x, labels) in enumerate(loader):
        x = x.to(device)
        labels = labels.to(device)

        with torch.no_grad():
            predictions = model(x)

        batch_size = x.shape[0]
        true_bboxes = cellboxes_to_boxes(labels)
        bboxes = cellboxes_to_boxes(predictions)

        for idx in range(batch_size):
            nms_boxes = nms(
                bboxes[idx],
                iou_threshold=iou_threshold,
                threshold=threshold,
                box_format=box_format,
            )

            for nms_box in nms_boxes:
                all_pred_boxes.append([image_idx] + nms_box)

            for box in true_bboxes[idx]:
                # many will get converted to 0 pred
                if box[1] > threshold:
                    all_true_boxes.append([image_idx] + box)

            image_idx += 1

    model.train()
    return all_pred_boxes, all_true_boxes
#找出最大置信度对应的框和最大可能的种类，输入[b,1470],输出[b,7,7,6]
#  输出归一化的[cls,p,x,y,w,h]
def convert_cellboxes(predictions, S=7):
    """
    Converts bounding boxes output from Yolo with
    an image split size of S into entire image ratios
    rather than relative to cell ratios. Tried to do this
    vectorized, but this resulted in quite difficult to read
    code... Use as a black box? Or implement a more intuitive,
    using 2 for loops iterating range(S) and convert them one
    by one, resulting in a slower but more readable implementation.
    """

    predictions = predictions.to("cpu")
    batch_size = predictions.shape[0]
    predictions = predictions.reshape(batch_size, 7, 7, 30)
    bboxes1 = predictions[..., 21:25]
    bboxes2 = predictions[..., 26:30]
    scores = torch.cat(
        (predictions[..., 20].unsqueeze(0), predictions[..., 25].unsqueeze(0)), dim=0
    )
    #[B,7,7,1]
    best_box = scores.argmax(0).unsqueeze(-1)
    #[b,7,7,4]
    best_boxes = bboxes1 * (1 - best_box) + best_box * bboxes2
   
    cell_indices = torch.arange(7).repeat(batch_size, 7, 1).unsqueeze(-1)
    x = 1 / S * (best_boxes[..., :1] + cell_indices)
    y = 1 / S * (best_boxes[..., 1:2] + cell_indices.permute(0, 2, 1, 3))
    w_h = 1 / S * best_boxes[..., 2:4]
    converted_bboxes = torch.cat((x, y, w_h), dim=-1)
    predicted_class = predictions[..., :20].argmax(-1).unsqueeze(-1)
    best_confidence = torch.max(predictions[..., 20], predictions[..., 25]).unsqueeze(
        -1
    )
    converted_preds = torch.cat(
        (predicted_class, best_confidence, converted_bboxes), dim=-1
    )

    return converted_preds
#转为list形式，输入[B,7,7,6],输出[B,49,6]
def cellboxes_to_boxes(out, S=7):
    converted_pred = convert_cellboxes(out).reshape(out.shape[0], S * S, -1)
    #将class转为整数
    converted_pred[..., 0] = converted_pred[..., 0].long()
    all_bboxes = []

    for ex_idx in range(out.shape[0]):
        bboxes = []

        for bbox_idx in range(S * S):
            bboxes.append([x.item() for x in converted_pred[ex_idx, bbox_idx, :]])
        all_bboxes.append(bboxes)

    return all_bboxes
#将预测值解码为真实的[x,y,w,h]
def decode(prediction,S=7,B=2,C=20,):
    """
    将 YOLOv1 输出转换为：

    [
        [class_id, score, x, y, width, height],
        ...
    ]
    """

    if prediction.ndim == 2:
        prediction = prediction[0]

    prediction = prediction.reshape(S,S,C + B * 5,)

    decoded_boxes = []

    for row in range(S):
        for col in range(S):
            cell = prediction[row, col]

            # 每个 Cell 共享一组类别概率
            class_score, class_id = torch.max(cell[:C],dim=0,)

            for box_index in range(B):
                start = C + box_index * 5

                object_confidence = cell[start]
                x_cell = cell[start + 1]
                y_cell = cell[start + 2]
                width_cell = cell[start + 3]
                height_cell = cell[start + 4]

                # Cell 坐标转换成整张图片的归一化坐标
                x = (x_cell + col) / S
                y = (y_cell + row) / S
                width = width_cell.abs() / S
                height = height_cell.abs() / S

                # 类别分数 × 目标置信度
                score = class_score * object_confidence

                decoded_boxes.append([
                    int(class_id.item()),
                    float(score.item()),
                    float(x.item()),
                    float(y.item()),
                    float(width.item()),
                    float(height.item()),
                ])

    return decoded_boxes
def save_checkpoint(state, filename="my_checkpoint.pth.tar"):
    print("=> Saving checkpoint")
    torch.save(state, filename)
def load_checkpoint(checkpoint, model, optimizer,device):
    print("=> Loading checkpoint")
    model.load_state_dict(checkpoint["state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer"])

data

In [3]:
VOC_CLASSES = [
    "aeroplane", "bicycle", "bird", "boat", "bottle",
    "bus", "car", "cat", "chair", "cow",
    "diningtable", "dog", "horse", "motorbike", "person",
    "pottedplant", "sheep", "sofa", "train", "tvmonitor",
]
CLASS_TO_IDX = {
    class_name: idx
    for idx, class_name in enumerate(VOC_CLASSES)
}
class VOCDataset(Dataset):
    def __init__(
        self,
        root,
        image_set="trainval",
        year="2007",
        S=7,
        B=2,
        C=20,
        transform=None,
        download=False,
    ):
        self.dataset = VOCDetection(
            root=root,
            year=year,
            image_set=image_set,
            download=download,
        )

        self.transform = transform
        self.S = S
        #B表示每个cell预测多少bounding box
        self.B = B
        self.C = C

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, index):
        image, target = self.dataset[index]

        annotation = target["annotation"]

        img_width = float(annotation["size"]["width"])
        img_height = float(annotation["size"]["height"])

        boxes = []

        objects = annotation.get("object", [])

        # 某些情况下只有一个 object 时可能不是 list
        if isinstance(objects, dict):
            objects = [objects]

        for obj in objects:
            class_name = obj["name"]
            class_label = CLASS_TO_IDX[class_name]

            bndbox = obj["bndbox"]

            xmin = float(bndbox["xmin"])
            ymin = float(bndbox["ymin"])
            xmax = float(bndbox["xmax"])
            ymax = float(bndbox["ymax"])

            # corners -> midpoint
            x = ((xmin + xmax) / 2) / img_width
            y = ((ymin + ymax) / 2) / img_height
            width = (xmax - xmin) / img_width
            height = (ymax - ymin) / img_height

            boxes.append([
                class_label,
                x,
                y,
                width,
                height,
            ])

        boxes = torch.tensor(boxes, dtype=torch.float32)

        if self.transform:
            image, boxes = self.transform(image),boxes

        # -----------------------------
        # 转成 YOLOv1 的 S×S×(C+5B)
        # -----------------------------

        label_matrix = torch.zeros(
            #生成7*7*30的全0矩阵
            (self.S, self.S, self.C + 5 * self.B),
            dtype=torch.float32,
        )

        for box in boxes:
            class_label, x, y, width, height = box.tolist()
            class_label = int(class_label)

            # 防止极端情况下 x/y == 1 导致索引越界
            x = min(x, 1 - 1e-6)
            y = min(y, 1 - 1e-6)

            # 确定目标中心属于哪个 cell
            i = int(self.S * y)
            j = int(self.S * x)

            # 中心点相对于当前 cell 的坐标
            x_cell = self.S * x - j
            y_cell = self.S * y - i

            # 宽高相对于 cell 尺寸
            width_cell = width * self.S
            height_cell = height * self.S

            # YOLOv1：一个 cell 只保留一个目标
            if label_matrix[i, j, 20] == 0:
                # objectness
                label_matrix[i, j, 20] = 1

                # bbox
                label_matrix[i, j, 21:25] = torch.tensor(
                    [
                        x_cell,
                        y_cell,
                        width_cell,
                        height_cell,
                    ],
                    dtype=torch.float32,
                )

                # class one-hot
                label_matrix[i, j, class_label] = 1

        return image, label_matrix

model

In [ ]:
class CNNBlock(nn.Module):
    def __init__(self, in_channels, out_channels, **kwargs):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, bias=False, **kwargs)
        self.batchnorm = nn.BatchNorm2d(out_channels)
        self.leakyrelu = nn.LeakyReLU(0.1)

    def forward(self, x):
        return self.leakyrelu(self.batchnorm(self.conv(x)))
class Yolov1(nn.Module):
    def __init__(self, S=7, B=2, C=20):
        super().__init__()

        self.model = nn.Sequential(
            # 输入:
            # (N, 3, 448, 448)
            CNNBlock(3, 64, kernel_size=7, stride=2, padding=3),
            # (N, 64, 224, 224)
            nn.MaxPool2d(kernel_size=2, stride=2),
            # (N, 64, 112, 112)
            
            CNNBlock(64, 192, kernel_size=3, stride=1, padding=1),
            # (N, 192, 112, 112)
            nn.MaxPool2d(kernel_size=2, stride=2),
            # (N, 192, 56, 56)

            CNNBlock(192, 128, kernel_size=1, stride=1, padding=0),
            # (N, 128, 56, 56)
            CNNBlock(128, 256, kernel_size=3, stride=1, padding=1),
            # (N, 256, 56, 56)

            CNNBlock(256, 256, kernel_size=1, stride=1, padding=0),
            # (N, 256, 56, 56)
            CNNBlock(256, 512, kernel_size=3, stride=1, padding=1),
            # (N, 512, 56, 56)
            nn.MaxPool2d(kernel_size=2, stride=2),
            # (N, 512, 28, 28)

            # 1*1压缩通道，减少参数量
            CNNBlock(512, 256, kernel_size=1, stride=1, padding=0),
            # (N, 256, 28, 28)
            CNNBlock(256, 512, kernel_size=3, stride=1, padding=1),
            # (N, 512, 28, 28)

            CNNBlock(512, 256, kernel_size=1, stride=1, padding=0),
            # (N, 256, 28, 28)
            CNNBlock(256, 512, kernel_size=3, stride=1, padding=1),
            # (N, 512, 28, 28)

            CNNBlock(512, 256, kernel_size=1, stride=1, padding=0),
            # (N, 256, 28, 28)
            CNNBlock(256, 512, kernel_size=3, stride=1, padding=1),
            # (N, 512, 28, 28)

            CNNBlock(512, 256, kernel_size=1, stride=1, padding=0),
            # (N, 256, 28, 28)
            CNNBlock(256, 512, kernel_size=3, stride=1, padding=1),
            # (N, 512, 28, 28)

            CNNBlock(512, 512, kernel_size=1, stride=1, padding=0),
            # (N, 512, 28, 28)
            CNNBlock(512, 1024, kernel_size=3, stride=1, padding=1),
            # (N, 1024, 28, 28)
            nn.MaxPool2d(kernel_size=2, stride=2),
            # (N, 1024, 14, 14)

            # repeat 2 times
            CNNBlock(1024, 512, kernel_size=1, stride=1, padding=0),
            # (N, 512, 14, 14)
            CNNBlock(512, 1024, kernel_size=3, stride=1, padding=1),
            # (N, 1024, 14, 14)

            CNNBlock(1024, 512, kernel_size=1, stride=1, padding=0),
            # (N, 512, 14, 14)
            CNNBlock(512, 1024, kernel_size=3, stride=1, padding=1),
            # (N, 1024, 14, 14)

            CNNBlock(1024, 1024, kernel_size=3, stride=1, padding=1),
            # (N, 1024, 14, 14)
            CNNBlock(1024, 1024, kernel_size=3, stride=2, padding=1),
            # (N, 1024, 7, 7)
            CNNBlock(1024, 1024, kernel_size=3, stride=1, padding=1),
            # (N, 1024, 7, 7)
            CNNBlock(1024, 1024, kernel_size=3, stride=1, padding=1),
            # (N, 1024, 7, 7)


            nn.Flatten(),
            # (N, 1024 * 7 * 7)
            nn.Linear(1024 * S * S, 4096),

            # (N, 4096)
            nn.Dropout(0.3),
            # (N, 4096)
            nn.LeakyReLU(0.1),
            # (N, 4096)
            nn.Linear(4096, S * S * (C + B * 5))
            # (N, 7*7*(20 + 2*5))
        )

    def forward(self, x):
        return self.model(x)

loss

In [5]:
#forward输入prediction为[B,7*7*30],输入target为[B,7,7,30]
class YoloLoss(nn.Module):
    """
    Calculate the loss for yolo (v1) model
    """

    def __init__(self, S=7, B=2, C=20):
        super().__init__()
        self.mse = nn.MSELoss(reduction="sum")

        """
        S is split size of image (in paper 7),
        B is number of boxes (in paper 2),
        C is number of classes (in paper and VOC dataset is 20),
        """
        self.S = S
        self.B = B
        self.C = C

        # These are from Yolo paper, signifying how much we should
        # pay loss for no object (noobj) and the box coordinates (coord)
        self.lambda_noobj = 0.5
        self.lambda_coord = 5

    def forward(self, predictions, target):
        # 输入 (B, 7*7*30) 转换为(B,7,7,30)
        predictions = predictions.reshape(-1, self.S, self.S, self.C + self.B * 5)

        # 计算iou,输出(2,B,7,7,1)
        iou_b1 = iou_cal(predictions[..., 21:25], target[..., 21:25])
        iou_b2 = iou_cal(predictions[..., 26:30], target[..., 21:25])
        ious = torch.cat([iou_b1.unsqueeze(0), iou_b2.unsqueeze(0)], dim=0)

        #获取更优的box,bestbox(B,7,7,1)
        iou_maxes, bestbox = torch.max(ious, dim=0)
        #cls,[b,7,7,1]
        exists_box = target[..., 20].unsqueeze(3)  # in paper this is Iobj_i

        # ======================== #
        #   FOR BOX COORDINATES    #
        # ======================== #

        #获得更优的预测框，[B,7*7，4]
        box_predictions = exists_box * (
            (
                bestbox * predictions[..., 26:30]
                + (1 - bestbox) * predictions[..., 21:25]
            )
        )

        box_targets = exists_box * target[..., 21:25]

        # Take sqrt of width, height of boxes to ensure that
        box_predictions[..., 2:4] = torch.sign(box_predictions[..., 2:4]) * torch.sqrt(
            torch.abs(box_predictions[..., 2:4] + 1e-6)
        )
        box_targets[..., 2:4] = torch.sqrt(box_targets[..., 2:4])

        #[b*7*7,4]
        box_loss = self.mse(
            torch.flatten(box_predictions, end_dim=-2),
            torch.flatten(box_targets, end_dim=-2),
        )

        # ==================== #
        #   FOR OBJECT LOSS    #
        # ==================== #

        # pred_box is the confidence score for the bbox with highest IoU
        pred_box = (
            bestbox * predictions[..., 25:26] + (1 - bestbox) * predictions[..., 20:21]
        )

        object_loss = self.mse(
            torch.flatten(exists_box * pred_box),
            torch.flatten(exists_box * target[..., 20:21]),
        )

        # ======================= #
        #   FOR NO OBJECT LOSS    #
        # ======================= #


        no_object_loss = self.mse(
            torch.flatten((1 - exists_box) * predictions[..., 20:21], start_dim=1),
            torch.flatten((1 - exists_box) * target[..., 20:21], start_dim=1),
        )

        no_object_loss += self.mse(
            torch.flatten((1 - exists_box) * predictions[..., 25:26], start_dim=1),
            torch.flatten((1 - exists_box) * target[..., 20:21], start_dim=1)
        )

        # ================== #
        #   FOR CLASS LOSS   #
        # ================== #

        class_loss = self.mse(
            torch.flatten(exists_box * predictions[..., :20], end_dim=-2,),
            torch.flatten(exists_box * target[..., :20], end_dim=-2,),
        )

        loss = (
            self.lambda_coord * box_loss  # first two rows in paper
            + object_loss  # third row in paper
            + self.lambda_noobj * no_object_loss  # forth row
            + class_loss  # fifth row
        )

        return loss

train

In [ ]:
# Hyperparameters etc. 
LEARNING_RATE = 2e-5
DEVICE = ("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 8 
WEIGHT_DECAY = 0
EPOCHS = 1
NUM_WORKERS = 2
LOAD_MODEL = False
USE_AMP = DEVICE == "cuda"
LOAD_MODEL_FILE = "./checkpoints/overfit.pth.tar"
data_path = '../Faster R-CNN/data'
os.makedirs("./checkpoints",exist_ok=True,)

def train_fn(train_loader, model, optimizer, loss_fn,scaler):
    model.train()
    loop = tqdm(train_loader, leave=True)
    mean_loss = []

    for batch_idx, (images, targets) in enumerate(loop):
        images = images.to(
            DEVICE,
            non_blocking=True,
        )

        targets = targets.to(
            DEVICE,
            non_blocking=True,
        )

        optimizer.zero_grad(set_to_none=True)

        # 只对前向传播和损失计算使用混合精度
        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=USE_AMP,
        ):
            predictions = model(images)
            loss = loss_fn(
                predictions,
                targets,
            )

        # 放大损失后进行反向传播
        scaler.scale(loss).backward()

        # 可选：梯度裁剪
        scaler.unscale_(optimizer)

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=10.0,
        )

        # 内部会检查梯度是否为 inf 或 NaN
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.detach().item()

        loop.set_postfix(
            loss=f"{loss.item():.2f}",
            scale=f"{scaler.get_scale():.0f}",
        )

    mean_loss = total_loss / len(train_loader)

    print(f"Mean loss: {mean_loss:.4f}")

    return mean_loss
def main():
    model = Yolov1(S=7,B=2,C=20,).to(DEVICE)
    optimizer = optim.Adam(
        model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )
    

    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=USE_AMP,
    )
    loss_fn = YoloLoss()
    transform = transforms.Compose([transforms.Resize((448, 448)),
                          transforms.ToTensor(),])

    if LOAD_MODEL:
        load_checkpoint(torch.load(LOAD_MODEL_FILE,map_location=DEVICE), model, optimizer,DEVICE)

    train_dataset = VOCDataset(
        root=data_path,
        image_set="trainval",
        year="2007",
        S=7,
        B=2,
        C=20,
        transform=transform,
        download=True,
    )
    test_dataset = VOCDataset(
        root=data_path,
        image_set="test",
        year="2007",
        S=7,
        B=2,
        C=20,
        transform=transform,
        download=True,
    )
    train_loader = DataLoader(
        dataset=train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True,
    )
    test_loader = DataLoader(
        dataset=test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
    )
    best_avg_prec= 0
    writer = SummaryWriter('./logs')
    for epoch in range(EPOCHS):
        mean_loss = train_fn(train_loader, model, optimizer, loss_fn,scaler = scaler)
        if (epoch+1) % 5 == 0:
            pred_boxes, target_boxes = get_bboxes(
                train_loader, model, iou_threshold=0.5, threshold=0.4,device=DEVICE
            )

            mean_avg_prec = mean_average_precision(
                pred_boxes, target_boxes, iou_threshold=0.5, box_format="midpoint"
            )
            print(f"Train mAP: {mean_avg_prec}")
      
            if mean_avg_prec > best_avg_prec:
                checkpoint = {
                "state_dict": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                }
        
                save_checkpoint(checkpoint, filename=LOAD_MODEL_FILE)
                best_avg_prec=mean_avg_prec
        writer.add_scalar('train/loss',mean_loss,epoch+1)

    writer.close()
if __name__ == "__main__":
    main()

KeyboardInterrupt: 

predict

In [ ]:
@torch.inference_mode()
def predict_image(
    model,
    image_path,
    device,
    score_threshold=0.20,
    iou_threshold=0.50,
):
    model.eval()

    original_image = Image.open(
        image_path
    ).convert("RGB")

    transform = transforms.Compose([
        transforms.Resize((448, 448)),
        transforms.ToTensor(),
    ])

    image_tensor = transform(
        original_image
    ).unsqueeze(0).to(device)

    prediction = model(image_tensor)

    decoded_boxes = decode(
        prediction,
        S=7,
        B=2,
        C=20,
    )

    # 直接调用文档中已有的 nms()
    final_boxes = nms(
        bboxes=decoded_boxes,
        iou_threshold=iou_threshold,
        threshold=score_threshold,
        box_format="midpoint",
    )

    # 限制坐标范围，避免绘图超出图片
    clipped_boxes = []

    for box in final_boxes:
        class_id, score, x, y, width, height = box

        x = min(max(x, 0.0), 1.0)
        y = min(max(y, 0.0), 1.0)
        width = min(max(width, 0.0), 1.0)
        height = min(max(height, 0.0), 1.0)

        clipped_boxes.append([
            class_id,
            score,
            x,
            y,
            width,
            height,
        ])

    return original_image, clipped_boxes
model = Yolov1(S=7,B=2,C=20,).to(DEVICE)

checkpoint = torch.load(
    LOAD_MODEL_FILE,
    map_location=DEVICE,
)
model.load_state_dict(checkpoint["state_dict"])
model.eval()
image, predicted_boxes = predict_image(
    model=model,
    image_path="./dog.jpg",
    device=DEVICE,
    score_threshold=0.10,
    iou_threshold=0.50,)
plot_image(
    image,
    predicted_boxes,
)